# Packages import


In [ ]:
import requests
import json
from bs4 import BeautifulSoup

# Ceneo scraper

1. Provide url adress of product's opinion page

In [ ]:
product_code="124893467"
page=1
url=f"https://www.ceneo.pl/{product_code}/opinie-{page}"
print(url)

https://www.ceneo.pl/124893467#tab=reviews


2. Send the request to provided url adress

In [39]:
response = requests.get(url=url)

3. If status code is OK, fetch all opinions from requested webpage

In [40]:
page_dom= BeautifulSoup(response.text, "html.parser")

In [41]:
product_name=page_dom.find("h1", {'class':'product-top__product-info__name'}).get_text()

In [42]:
product_name=page_dom.select_one("h1", {'class':'product-top__product-info__name'}).get_text()

4. For all fetched opinions, parse them to extract relevant data 


In [43]:
opinions=page_dom.find_all('div',{'class':'js_product-review'})

In [44]:
opinions=page_dom.select('div.js_product-review')   
print(type(opinions))
print(len(opinions))

<class 'bs4.element.ResultSet'>
11


In [45]:
opinions=page_dom.select('div.js_product-review')
print(type(opinions))   
print(len(opinions))

<class 'bs4.element.ResultSet'>
11


In [ ]:
opinions=page_dom.select('div.js_product-rewiev:not(.user-post--highlight)')
print(type(opinions))
print(len(opinions))

In [ ]:
[opinion for opinion in page_dom.find_all('div',{'class':'js_product-review'}) if "user-post--highlight" not in  opinion.get("class",[])]

5. For all fetched opinions, parse them to extract relevant data 

In [ ]:
all_opinions=[]
for opinion in opinions:
    single_opinion={
        'opinion_id':opinion['data-entry-id'],
        'author':opinion.select_one('span.user-post__author-name').get_text(),
        'reccomendation':opinion.select_one('span.user-post__author-recommendation>em').get_text(),
        'score':opinion.select_one('span.user-post__score-count').get_text(),
        'content':opinion.select_one('div.user-post__text').get_text(),
        'pros':[p.get_text() for p in opinion.select('div.review-feature__item--positive')],
        'cons':[c.get_text() for c in opinion.select('div.review-feature__item--negattive')],
        'like':opinion.select_one('div.review-feature__item--negattive').get_text(),
        'dislike':opinion.select_one('button.vote.yes > span').get_text(),
        'publish_date':opinion.select_one('	span.user-post__published > time:nth-child(1)[datetime]').get_text(),
        'purchase_date':opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]').get_text() if opinion.select_one('span.user-post__published > time:nth-child(2)[datetime]') else None,
    }
    all_opinions.append(single_opinion)

6. Check if there is next page with opinions

In [ ]:
next= True if page_dom.select_one('button.pagination__next') else False
if next:page+=1

8. Save obtained opinions

In [ ]:



with open(f"./opinions/{product_code}.json,'w',encoding='UTF-8'") as jf:
    json.dump(all_opinions,jf,indent=4,ensure_ascii=False)